# 01 - Sources and retrieval

Here I check what we pulled from each source and whether the pull is complete, and at the end I size the
problem straight from the raw trip files, before any cleaning.

The files are downloaded by the pipeline (`python -m pipeline.run --months 2026-04 2026-05 2026-06`), so run
that first. This notebook only reads what it saved in `data/raw/` and `data/raw_manifest.json`.

Two retrieval modes:

1. Files over HTTPS: the monthly yellow taxi parquet files and the taxi zone lookup CSV (TLC, via CloudFront).
2. A JSON API: the Open-Meteo historical weather archive, one response per month.

DuckDB SQL is then used to query and transform the files (here and in the pipeline).


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so we can import the pipeline package

import duckdb
import pandas as pd
import pyarrow.parquet as pq
import requests
from pipeline import config, ingest

pd.set_option("display.width", 200)
con = duckdb.connect()

## Raw manifest

Every raw file gets an entry with its size on disk, the `Content-Length` the server sent (when the response
wasn't compressed), Last-Modified and a sha256. On a rerun the pipeline only skips a download if size and
sha256 still match.


In [2]:
manifest = json.loads(config.MANIFEST_PATH.read_text())
m = pd.DataFrame(manifest["files"]).T
files = m[m.index.str.startswith(("yellow", "taxi"))].copy()
files["size_matches_http"] = (files["bytes"] == files["http_content_length"]).where(files["http_content_length"].notna())
files["sha256"] = files["sha256"].str[:16] + "..."
files[["bytes", "http_content_length", "size_matches_http", "parquet_num_rows", "last_modified", "sha256"]]

,bytes,http_content_length,size_matches_http,parquet_num_rows,last_modified,sha256
taxi_zone_lookup.csv,12331,None,NaN,NaN,"Thu, 22 Feb 2024 21:33:00 GMT",1a99e105092230f8...
yellow_tripdata_2026-04.parquet,64818115,64818115,True,3831240,"Fri, 05 Jun 2026 20:44:34 GMT",e03b82d5722a6b21...
yellow_tripdata_2026-05.parquet,69699174,69699174,True,4090836,"Fri, 26 Jun 2026 21:07:20 GMT",9aa5a1609e2bf07d...
yellow_tripdata_2026-06.parquet,65465637,65465637,True,3837248,"Thu, 17 Sep 2026 18:34:43 GMT",f340c62a18885c56...


The zone CSV comes back gzip-compressed, so its `Content-Length` is the compressed size and isn't recorded;
for that file I rely on the row count (265 zones, checked below) and the sha256.

The `last_modified` dates don't follow a neat monthly rhythm (April's file is dated 5 Jun 2026, June's is
dated 17 Sep 2026), so a month could be replaced after I downloaded it. On a rerun the pipeline sends a HEAD
request for each trip file and downloads it again if Last-Modified changed. If that request fails (no network)
it keeps the local copy and logs a warning. `--force-download` fetches everything again regardless.


In [3]:
weather = m[m.index.str.startswith("open_meteo")].copy()
weather["sha256"] = weather["sha256"].str[:16] + "..."
weather[["bytes", "hours", "retrieved_at", "sha256"]]

,bytes,hours,retrieved_at,sha256
open_meteo_2026-04.json,20874,720,2026-09-24T20:38:08Z,b91cfe4a8ba80be7...
open_meteo_2026-05.json,21871,744,2026-09-24T20:38:17Z,db52d35a826289fd...
open_meteo_2026-06.json,21205,720,2026-09-24T20:38:27Z,fe982e71ba16165a...


## Is the trip retrieval complete?

The parquet footer stores how many rows the publisher wrote. I compare that with what DuckDB actually reads
back, check that the pickup dates cover every day of the month, and compare each month's size with the month
before (the pipeline warns if it moves by more than 15%).


In [4]:
rows = []
for f in sorted((config.RAW_DIR / "trips").glob("yellow_tripdata_*.parquet")):
    month = f.stem[-7:]
    footer = pq.ParquetFile(f).metadata.num_rows
    loaded, days, in_month = con.execute(f'''
        SELECT count(*),
               count(DISTINCT CAST(tpep_pickup_datetime AS DATE)) FILTER (WHERE strftime(tpep_pickup_datetime, '%Y-%m') = '{month}'),
               count(*) FILTER (WHERE strftime(tpep_pickup_datetime, '%Y-%m') = '{month}')
        FROM read_parquet('{f}')''').fetchone()
    rows.append({"month": month, "rows_in_footer": footer, "rows_read": loaded, "match": footer == loaded,
                 "days_with_pickups": days, "pickups_outside_month": loaded - in_month})
completeness = pd.DataFrame(rows)
completeness["change_vs_previous_month"] = completeness["rows_read"].pct_change().round(3)
completeness

,month,rows_in_footer,rows_read,match,days_with_pickups,pickups_outside_month,change_vs_previous_month
0,2026-04,3831240,3831240,True,30,12,NaN
1,2026-05,4090836,4090836,True,31,14,0.068
2,2026-06,3837248,3837248,True,30,17,-0.062


## Schema check across months

The files are supposed to share one schema. They don't quite: June has an extra column.


In [5]:
schemas = {f.stem[-7:]: pq.ParquetFile(f).schema_arrow.names
           for f in sorted((config.RAW_DIR / "trips").glob("yellow_tripdata_*.parquet"))}
all_cols = sorted(set(c for cols in schemas.values() for c in cols))
pd.DataFrame({m: [c in cols for c in all_cols] for m, cols in schemas.items()}, index=all_cols).loc[
    lambda d: ~d.all(axis=1)]

,2026-04,2026-05,2026-06
request_source,False,False,True


In [6]:
june = config.RAW_DIR / "trips" / "yellow_tripdata_2026-06.parquet"
con.sql(f"SELECT request_source, count(*) AS trips FROM read_parquet('{june}') GROUP BY 1 ORDER BY 2 DESC").df()

,request_source,trips
0,NaN,2824068
1,HV0003,927698
2,A,76711
3,EH0004,8107
4,CC,664


`request_source` is not in the data dictionary (March 2025 version). `HV0003` looks like a high-volume
for-hire license code, so these may be trips booked through a ride-hail app, but the dictionary doesn't say
that, so it goes on the unknowns list. The pipeline only reads the 13 columns it needs, and check S01 reports
the extra column every run.

## Taxi zone lookup


In [7]:
zones = pd.read_csv(config.RAW_DIR / "taxi_zone_lookup.csv")
print(len(zones), "zones")
print(zones[zones.LocationID.isin([132, 264, 265])].to_string(index=False))
print("Manhattan zones:", (zones.Borough == "Manhattan").sum())

265 zones
 LocationID Borough           Zone service_zone
        132  Queens    JFK Airport     Airports
        264 Unknown            NaN          NaN
        265     NaN Outside of NYC          NaN
Manhattan zones: 69


## Weather API responses

Open-Meteo returns hourly values in local New York time. I asked for `precipitation` and `temperature_2m` at
the JFK point. Complete means one row per hour of the month and no nulls.


In [8]:
wx_rows = []
for f in sorted((config.RAW_DIR / "weather").glob("open_meteo_*.json")):
    month = f.stem[-7:]
    payload = json.loads(f.read_text())
    ok, msg = ingest.check_weather(payload, month)
    h = pd.DataFrame(payload["hourly"])
    wx_rows.append({"month": month, "grid_lat": payload["latitude"], "grid_lon": payload["longitude"],
                    "hours": len(h), "expected": ingest.expected_hours(month), "complete": ok,
                    "wet_hours (>=0.5 mm)": int((h.precipitation >= config.WET_HOUR_MM).sum()),
                    "total_precip_mm": round(h.precipitation.sum(), 1)})
pd.DataFrame(wx_rows)

,month,grid_lat,grid_lon,hours,expected,complete,wet_hours (>=0.5 mm),total_precip_mm
0,2026-04,40.667835,-73.81021,720,720,True,42,75.0
1,2026-05,40.667835,-73.81021,744,744,True,60,95.6
2,2026-06,40.667835,-73.81021,720,720,True,46,98.2


What counts as "expected hours" on a daylight-saving change day? The local clock skips or repeats an hour,
so I asked the API for the two most recent change days to see what it returns:


In [9]:
for day in ["2026-03-08", "2025-11-02"]:
    try:
        r = requests.get(config.WEATHER_URL, timeout=60, params={
            "latitude": config.WEATHER_LAT, "longitude": config.WEATHER_LON, "start_date": day, "end_date": day,
            "hourly": "precipitation", "timezone": config.TIMEZONE})
        times = r.json()["hourly"]["time"]
        print(day, len(times), "rows, first four:", times[:4])
    except requests.RequestException as e:
        print(day, "could not reach the weather API, skipping this check:", e)

2026-03-08 24 rows, first four: ['2026-03-08T00:00', '2026-03-08T01:00', '2026-03-08T02:00', '2026-03-08T03:00']


2025-11-02 24 rows, first four: ['2025-11-02T00:00', '2025-11-02T01:00', '2025-11-02T02:00', '2025-11-02T03:00']


Both days come back with 24 rows, so `expected_hours` is simply days x 24. That doesn't matter for April to
June, but it would for March or November. The grid cell Open-Meteo snapped to is a few km from the terminals,
which is fine for "was it raining around JFK in that hour" and says nothing about the route into Manhattan.

## Sizing the problem from the raw data

Before any cleaning or rules: how many trips go from JFK (zone 132) to a Manhattan zone, and how many take
over an hour?


In [10]:
glob = str(config.RAW_DIR / "trips" / "yellow_tripdata_*.parquet")
con.sql(f'''
    WITH t AS (
        SELECT substr(filename, -15, 7) AS month, RatecodeID,
               date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime) / 60.0 AS minutes
        FROM read_parquet('{glob}', filename = true, union_by_name = true)
        WHERE PULocationID = 132
          AND DOLocationID IN (SELECT LocationID FROM read_csv('{config.RAW_DIR / "taxi_zone_lookup.csv"}')
                               WHERE Borough = 'Manhattan'))
    SELECT coalesce(month, 'all') AS month, count(*) AS trips,
           round(count(*) FILTER (WHERE RatecodeID = 2) / count(*), 3) AS share_rate_code_2,
           round(count(*) FILTER (WHERE minutes > 60) / count(*), 3) AS share_over_60_min,
           round(median(minutes), 1) AS median_min
    FROM t GROUP BY ROLLUP (month) ORDER BY month''').df()

,month,trips,share_rate_code_2,share_over_60_min,median_min
0,2026-04,69945,0.953,0.281,50.1
1,2026-05,72049,0.950,0.396,54.5
2,2026-06,70694,0.954,0.353,53.3
3,all,212688,0.952,0.344,52.5


So about 70,000 JFK -> Manhattan trips a month, around 95% on the flat fare, and roughly a third
(34.4% before any rules) take over an hour. The median is about 52 minutes. That's enough to say there's a
real problem: a lot of $70 fares are paying the driver less than $70 per hour. How it splits by hour, and
whether the dirty rows change the picture, is in notebooks 02 and 03.

## Retrieval checks in short

- All three trip files read back with exactly the rows in their footer, every day has pickups, only 12-17
  pickups per file sit outside the file's month, and month-to-month volume moves by about 6-7%.
- Parquet file sizes on disk match the `Content-Length` the server sent, and every file has a sha256 in the
  manifest, so a rerun can show it's using the same bytes.
- The weather responses are complete for all three months.
- Raw files are never edited. Everything downstream reads from `data/raw/` and writes somewhere else.
